In [ ]:
import zipfile, os
from pathlib import Path

ZIP_PATH = Path("/content/datathon-2026-round-1.zip")
DATA_DIR = Path("/content/datathon_data")

DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

print("Extracted files:")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print(p)

Extracted files:
/content/datathon_data/baseline.ipynb
/content/datathon_data/customers.csv
/content/datathon_data/geography.csv
/content/datathon_data/inventory.csv
/content/datathon_data/order_items.csv
/content/datathon_data/orders.csv
/content/datathon_data/payments.csv
/content/datathon_data/products.csv
/content/datathon_data/promotions.csv
/content/datathon_data/returns.csv
/content/datathon_data/reviews.csv
/content/datathon_data/sales.csv
/content/datathon_data/sample_submission.csv
/content/datathon_data/shipments.csv
/content/datathon_data/web_traffic.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/datathon_data")

def find_file(name):
    matches = list(DATA_DIR.rglob(name))
    if len(matches) == 0:
        raise FileNotFoundError(f"Cannot find {name}")
    return matches[0]

orders = pd.read_csv(find_file("orders.csv"), parse_dates=["order_date"])
order_items = pd.read_csv(find_file("order_items.csv"))
products = pd.read_csv(find_file("products.csv"))
returns = pd.read_csv(find_file("returns.csv"), parse_dates=["return_date"])
web = pd.read_csv(find_file("web_traffic.csv"), parse_dates=["date"])
customers = pd.read_csv(find_file("customers.csv"), parse_dates=["signup_date"])
payments = pd.read_csv(find_file("payments.csv"))
geography = pd.read_csv(find_file("geography.csv"))

answers = {}

# Q1
tmp = orders.sort_values(["customer_id", "order_date"])
tmp["gap"] = tmp.groupby("customer_id")["order_date"].diff().dt.days
q1 = tmp["gap"].dropna().median()
answers["Q1"] = q1

# Q2
prod = products.copy()
prod["gross_margin_rate"] = (prod["price"] - prod["cogs"]) / prod["price"]
answers["Q2"] = prod.groupby("segment")["gross_margin_rate"].mean().sort_values(ascending=False)

# Q3
ret_prod = returns.merge(products[["product_id", "category"]], on="product_id", how="left")
answers["Q3"] = ret_prod[ret_prod["category"] == "Streetwear"]["return_reason"].value_counts()

# Q4
answers["Q4"] = web.groupby("traffic_source")["bounce_rate"].mean().sort_values()

# Q5
answers["Q5"] = order_items["promo_id"].notna().mean() * 100

# Q6
cust_orders = orders.groupby("customer_id").size().rename("num_orders").reset_index()
cust2 = customers.merge(cust_orders, on="customer_id", how="left")
cust2["num_orders"] = cust2["num_orders"].fillna(0)
answers["Q6"] = cust2[cust2["age_group"].notna()].groupby("age_group")["num_orders"].mean().sort_values(ascending=False)

# Q7
# Because sales.csv only has Date, Revenue, COGS, compute by reconstructing revenue by region from orders + order_items.
oi = order_items.merge(orders[["order_id", "zip"]], on="order_id", how="left")
oi = oi.merge(geography[["zip", "region"]], on="zip", how="left")

oi["line_revenue"] = oi["quantity"] * oi["unit_price"] - oi["discount_amount"]
answers["Q7"] = oi.groupby("region")["line_revenue"].sum().sort_values(ascending=False)

# Q8
answers["Q8"] = orders[orders["order_status"] == "cancelled"]["payment_method"].value_counts()

# Q9
oi_prod = order_items.merge(products[["product_id", "size"]], on="product_id", how="left")
ret_size = returns.merge(products[["product_id", "size"]], on="product_id", how="left")

den = oi_prod[oi_prod["size"].isin(["S", "M", "L", "XL"])].groupby("size").size()
num = ret_size[ret_size["size"].isin(["S", "M", "L", "XL"])].groupby("size").size()

answers["Q9"] = (num / den).sort_values(ascending=False)

# Q10
answers["Q10"] = payments.groupby("installments")["payment_value"].mean().sort_values(ascending=False)

for k, v in answers.items():
    print("\n" + "="*20)
    print(k)
    print(v)

/tmp/ipykernel_3035/804908765.py:14: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(find_file("order_items.csv"))



Q1
144.0

Q2
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin_rate, dtype: float64

Q3
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Q4
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Q5
38.663493169565214

Q6
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: num_orders, dtype: float64

Q7
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: line_revenue, dtype: float64

Q8
payment_method
credit_card      28452
cod              15468
paypal            7

Q1: 144 ngày

Q2: Standard

Q3: wrong_size

Q4: email_campaign

Q5: 39%

Q6: 55+

Q7: East

Q8: Credit Card

Q9: S

Q10: 6 kỳ
